<a href="https://colab.research.google.com/github/weathon/neg/blob/main/NAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [150]:
!git clone https://github.com/ChenDarYen/Normalized-Attention-Guidance.git

fatal: destination path 'Normalized-Attention-Guidance' already exists and is not an empty directory.


In [151]:
!pip install image-reward clip -qq

In [152]:
!pip install -r Normalized-Attention-Guidance/requirements.txt -qq

In [153]:
import sys
sys.path.append("Normalized-Attention-Guidance")

In [154]:
from huggingface_hub import login

In [155]:
import ImageReward as reward
reward_model = reward.load("ImageReward-v1.0")

load checkpoint from /root/.cache/ImageReward/ImageReward.pt
checkpoint loaded


In [156]:
import math
from typing import Optional

import torch
import torch.nn.functional as F

from diffusers.models.attention_processor import Attention
from diffusers.models.embeddings import apply_rotary_emb
import pylab
import random

class NAGFluxAttnProcessor2_0:
    """Attention processor used typically in processing the SD3-like self-attention projections."""

    def __init__(
            self,
            nag_scale: float = 1.0,
            nag_tau=2.5,
            nag_alpha=0.25,
            encoder_hidden_states_length: int = None,
            pos_prompt_length: int = None,
            neg_prompt_length: int = None,
    ):
        if not hasattr(F, "scaled_dot_product_attention"):
            raise ImportError("FluxAttnProcessor2_0 requires PyTorch 2.0, to use it, please upgrade PyTorch to 2.0.")
        self.nag_scale = nag_scale
        self.nag_tau = nag_tau
        self.nag_alpha = nag_alpha
        self.encoder_hidden_states_length = encoder_hidden_states_length
        self.pos_prompt_length = pos_prompt_length
        self.neg_prompt_length = neg_prompt_length

    def __call__(
        self,
        attn: Attention,
        hidden_states: torch.FloatTensor,
        encoder_hidden_states: torch.FloatTensor = None,
        attention_mask: Optional[torch.FloatTensor] = None,
        image_rotary_emb: Optional[torch.Tensor] = None,
    ) -> torch.FloatTensor:
        batch_size, _, _ = hidden_states.shape if encoder_hidden_states is None else encoder_hidden_states.shape
        # print(hidden_states.shape)
        if self.nag_scale > 1.:
            if encoder_hidden_states is not None:
                assert len(hidden_states) == batch_size * 0.5
            apply_guidance = True
        else:
            apply_guidance = False

        # `sample` projections.
        query = attn.to_q(hidden_states)
        key = attn.to_k(hidden_states)
        value = attn.to_v(hidden_states)

        # attention
        if apply_guidance and encoder_hidden_states is not None:
            query = query.tile(2, 1, 1)
            key = key.tile(2, 1, 1)
            value = value.tile(2, 1, 1)

        inner_dim = key.shape[-1]
        head_dim = inner_dim // attn.heads

        query = query.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        key = key.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)
        value = value.view(batch_size, -1, attn.heads, head_dim).transpose(1, 2)

        if attn.norm_q is not None:
            query = attn.norm_q(query)
        if attn.norm_k is not None:
            key = attn.norm_k(key)

        # the attention in FluxSingleTransformerBlock does not use `encoder_hidden_states`
        if encoder_hidden_states is not None:
            # `context` projections.
            encoder_hidden_states_query_proj = attn.add_q_proj(encoder_hidden_states)
            encoder_hidden_states_key_proj = attn.add_k_proj(encoder_hidden_states)
            encoder_hidden_states_value_proj = attn.add_v_proj(encoder_hidden_states)

            encoder_hidden_states_query_proj = encoder_hidden_states_query_proj.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            encoder_hidden_states_key_proj = encoder_hidden_states_key_proj.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)
            encoder_hidden_states_value_proj = encoder_hidden_states_value_proj.view(
                batch_size, -1, attn.heads, head_dim
            ).transpose(1, 2)

            if attn.norm_added_q is not None:
                encoder_hidden_states_query_proj = attn.norm_added_q(encoder_hidden_states_query_proj)
            if attn.norm_added_k is not None:
                encoder_hidden_states_key_proj = attn.norm_added_k(encoder_hidden_states_key_proj)

            query = torch.cat([encoder_hidden_states_query_proj, query], dim=2)
            key = torch.cat([encoder_hidden_states_key_proj, key], dim=2)
            value = torch.cat([encoder_hidden_states_value_proj, value], dim=2)

            encoder_hidden_states_length = encoder_hidden_states.shape[1]

        else:
            assert self.encoder_hidden_states_length is not None
            encoder_hidden_states_length = self.encoder_hidden_states_length

        if image_rotary_emb is not None:
            query = apply_rotary_emb(query, image_rotary_emb)
            key = apply_rotary_emb(key, image_rotary_emb)

        if not apply_guidance:
            hidden_states = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0, is_causal=False)
            hidden_states = hidden_states.transpose(1, 2).reshape(batch_size, -1, attn.heads * head_dim)
            hidden_states = hidden_states.to(query.dtype)

        else:
            origin_batch_size = batch_size // 2
            query, query_negative = torch.chunk(query, 2, dim=0)
            key, key_negative = torch.chunk(key, 2, dim=0)
            value, value_negative = torch.chunk(value, 2, dim=0)
            attn_mask = None
            if encoder_hidden_states is not None:
              encoder_neg_query = query_negative[:,:,:encoder_hidden_states_length]
              encoder_neg_key = key_negative[:,:,:encoder_hidden_states_length]
              encoder_neg_value = value_negative[:,:,:encoder_hidden_states_length]
              query_pos_neg = torch.cat([encoder_neg_query, query], dim=2)
              key_pos_neg = torch.cat([encoder_neg_key, key], dim=2)
              value_pos_neg = torch.cat([- 2 * encoder_neg_value, value], dim=2)

              attn_mask = torch.zeros((query_pos_neg.shape[0], query_pos_neg.shape[-2], value_pos_neg.shape[-2])).cuda().bfloat16()
              # mask between positive and negative prompts
              l = encoder_hidden_states_length
              attn_mask[:,0:l,l:l*2] = -1000
              attn_mask[:,l:l*2,0:l] = -1000
              attn_mask[:,:,self.neg_prompt_length:l+self.neg_prompt_length] = -1000


              hidden_states_regular = F.scaled_dot_product_attention(query, key, value, dropout_p=0.0, is_causal=False)
              hidden_states_regular = hidden_states_regular.transpose(1, 2).reshape(origin_batch_size, -1, attn.heads * head_dim)
              hidden_states_regular = hidden_states_regular.to(query.dtype)

            else:
              query_pos_neg, key_pos_neg, value_pos_neg = query, key, value

            hidden_state_pure_negative = F.scaled_dot_product_attention(query_negative, key_negative, value_negative, dropout_p=0.0, is_causal=False)
            hidden_state_pure_negative = hidden_state_pure_negative.transpose(1, 2).reshape(origin_batch_size, -1, attn.heads * head_dim)
            hidden_state_pure_negative = hidden_state_pure_negative.to(query.dtype)

            hidden_states = F.scaled_dot_product_attention(query_pos_neg, key_pos_neg, value_pos_neg, dropout_p=0.0, is_causal=False, attn_mask=attn_mask)
            hidden_states = hidden_states.transpose(1, 2).reshape(origin_batch_size, -1, attn.heads * head_dim)
            hidden_states = hidden_states.to(query.dtype)


            if encoder_hidden_states is not None:
              # hidden_states[:,encoder_hidden_states_length:2*encoder_hidden_states_length] -= hidden_states[:,:encoder_hidden_states_length]
              hidden_states = hidden_states[:,encoder_hidden_states_length:]
              assert hidden_states_regular.shape == hidden_states.shape == hidden_state_pure_negative.shape

        if encoder_hidden_states is not None:
            encoder_hidden_states, hidden_states = (
                hidden_states[:, : encoder_hidden_states.shape[1]],
                hidden_states[:, encoder_hidden_states.shape[1] :],
            )

            if apply_guidance:
                encoder_hidden_states_regular, hidden_states_regular = (
                    hidden_states_regular[:, : encoder_hidden_states.shape[1]],
                    hidden_states_regular[:, encoder_hidden_states.shape[1]:],
                )

                encoder_hidden_state_pure_negative, hidden_state_pure_negative = (
                    hidden_state_pure_negative[:, : encoder_hidden_states.shape[1]],
                    hidden_state_pure_negative[:, encoder_hidden_states.shape[1]:],
                )

                hidden_states = hidden_states * self.nag_scale - hidden_state_pure_negative * (self.nag_scale - 1)

                norm_positive = torch.norm(hidden_states_regular, p=1, dim=-1, keepdim=True).expand(*hidden_states_regular.shape)
                norm_guidance = torch.norm(hidden_states, p=1, dim=-1, keepdim=True).expand(*hidden_states_regular.shape)

                scale = norm_guidance / norm_positive
                hidden_states = hidden_states * torch.minimum(scale, scale.new_ones(1) * self.nag_tau) / scale

                hidden_states = hidden_states * self.nag_alpha + hidden_states_regular * (1 - self.nag_alpha)

                encoder_hidden_states = torch.cat((encoder_hidden_states, encoder_hidden_state_pure_negative), dim=0)
                # encoder_hidden_states = torch.cat((encoder_hidden_states, encoder_hidden_states_regular), dim=0)

            hidden_states = attn.to_out[0](hidden_states)
            hidden_states = attn.to_out[1](hidden_states)

            encoder_hidden_states = attn.to_add_out(encoder_hidden_states)
            # print(hidden_states.shape, encoder_hidden_states.shape)
            return hidden_states, encoder_hidden_states

        else:
            if apply_guidance:
                image_hidden_state_pure_negative = hidden_state_pure_negative[:, encoder_hidden_states_length:]
                image_hidden_states = hidden_states[:, encoder_hidden_states_length:]

                image_hidden_states_guidance = image_hidden_states * self.nag_scale - image_hidden_state_pure_negative * (self.nag_scale - 1)
                norm_positive = torch.norm(image_hidden_states, p=2, dim=-1, keepdim=True).expand(*image_hidden_states.shape)
                norm_guidance = torch.norm(image_hidden_states_guidance, p=2, dim=-1, keepdim=True).expand(*image_hidden_states.shape)

                scale = norm_guidance / norm_positive
                image_hidden_states_guidance = image_hidden_states_guidance * torch.minimum(scale, scale.new_ones(1) * self.nag_tau) / scale
                # scale = torch.nan_to_num(scale, 10)
                # image_hidden_states_guidance[scale > self.nag_tau] = image_hidden_states_guidance[scale > self.nag_tau] / (norm_guidance[scale > self.nag_tau] + 1e-7) * norm_positive[scale > self.nag_tau] * self.nag_tau

                image_hidden_states = image_hidden_states_guidance * self.nag_alpha + image_hidden_states * (1 - self.nag_alpha)

                hidden_state_pure_negative[:, encoder_hidden_states_length:] = image_hidden_states
                hidden_states[:, encoder_hidden_states_length:] = image_hidden_states
                hidden_states = torch.cat((hidden_states, hidden_state_pure_negative), dim=0)
            return hidden_states

In [157]:
import torch
from src.pipeline_flux_nag import NAGFluxPipeline
from src.transformer_flux import NAGFluxTransformer2DModel
from src.attention_flux_nag import NAGFluxAttnProcessor2_0 as NAGFluxAttnProcessor2_0_vanilla

transformer = NAGFluxTransformer2DModel.from_pretrained(
    "black-forest-labs/FLUX.1-schnell",
    subfolder="transformer",
    torch_dtype=torch.bfloat16,
)

if not "pipe" in locals():
  pipe = NAGFluxPipeline.from_pretrained(
      "black-forest-labs/FLUX.1-schnell",
      transformer=transformer,
      torch_dtype=torch.bfloat16,
  )
  pipe.to("cuda")

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [158]:
prompts = [
    {
        "pos": "a bathroom in a house",
        "neg": "modern"
    },
    {
        "pos": "a living room in a house",
        "neg": "clean"
    },
    {
        "pos": "a house made from wood",
        "neg": "windows"
    },
    {
        "pos": "a cup of soda in a container",
        "neg": "round cylinder"
    },
    {
        "pos": "a cup of soda in a container",
        "neg": "straw"
    },
    {
        "pos": "a wireless earphone on the table",
        "neg": "plastic"
    },
    {
        "pos": "a pair of shoes on the ground",
        "neg": "fabric"
    },
    {
      "pos": "a human standing in the office with a suit",
      "neg": "a tie"
    },
    {
        "pos": "a human drink soda",
        "neg": "dark liquid"
    },
    {
        "pos": "A fork slices through a soft, fluffy cake, which is beautifully frosted and placed on a fine porcelain plate, illuminated by warm, ambient lighting for a cozy, inviting atmosphere.",
        "neg": "metallic"
    },
    {
        "pos": "A modern hotel room featuring a stylish upholstered headboard, and soft ambient lighting. The bed is centrally positioned in the scene, with tasteful decor creating an inviting and tranquil atmosphere suitable for relaxation.",
        "neg": "pillows"
    },
    {
        "pos": "portrait of AI Researcher",
        "neg": "glasses"
    },
    {
        "pos": "A laptop is sitting silently on a table",
        "neg": "metallic"
    },
    {
        "pos": "A water bottle is on the table",
        "neg": "plastic"
    },
    {
        "pos": "a disposable fork besides a lunch box",
        "neg": "wooden"
    },
    {
        "pos": "a disposable fork besides a lunch box",
        "neg": "plastic"
    },
    {
        "pos": "a cooked chicken in a fast food box",
        "neg": "paper box"
    },
    {
        "pos": "a parking lot with cars during afternoon",
        "neg": "dark color cars"
    },
    {
        "pos": "a iPad on a desk",
        "neg": "screen lit"
    },
    {
        "pos": "a iPad on a desk",
        "neg": "screen unlit"
    },
    {
        "pos": "a paint of a person",
        "neg": "colored"
    },
    {
        "pos": "A short fence stood beside a teardrop fig, on a carpet of grass",
        "neg": "fluffy"
    },
    {
        "pos": "A teardrop kiwi and a round bagel sit on a table",
        "neg": "wooden"
    },
    {
        "pos": "a HDMI cable laying on the ground",
        "neg": "rubber"
    }
]

In [159]:
import wandb
from google.colab import userdata
# wandb.login(userdata.get('wandb'))

In [160]:
def _set_nag_attn_processor(
        self,
        nag_scale,
        encoder_hidden_states_length,
        nag_tau=5,
        nag_alpha=0.8,
):
    attn_procs = {}
    for name in self.transformer.attn_processors.keys():
        attn_procs[name] = NAGFluxAttnProcessor2_0(
            nag_scale=nag_scale,
            nag_tau=nag_tau,
            nag_alpha=nag_alpha,
            encoder_hidden_states_length=encoder_hidden_states_length,
            neg_prompt_length=5
        )
    self.transformer.set_attn_processor(attn_procs)

def _set_nag_attn_processor_vanilla(
        self,
        nag_scale,
        encoder_hidden_states_length,
        nag_tau=5,
        nag_alpha=0.8,
):
    attn_procs = {}
    for name in self.transformer.attn_processors.keys():
        attn_procs[name] = NAGFluxAttnProcessor2_0_vanilla(
            nag_scale=nag_scale,
            nag_tau=nag_tau,
            nag_alpha=nag_alpha,
            encoder_hidden_states_length=encoder_hidden_states_length,
        )
    self.transformer.set_attn_processor(attn_procs)


In [161]:
import os
import json
import base64
import io
from PIL import Image
from pydantic import BaseModel
from openai import OpenAI
from google.colab import userdata

client = OpenAI(api_key=userdata.get('key'))
class Score(BaseModel):
    first_image_positive: float
    first_image_negative: float
    first_image_quality: float
    second_image_positive: float
    second_image_negative: float
    second_image_quality: float


def ask_gpt(image1: Image.Image, image2: Image.Image, pos: str, neg: str) -> list[Score]:
    if random.random() > 0.5:
      image1, image2 = image2, image1
      swapped = True
    else:
      swapped = False

    # Encode both images
    buf1 = io.BytesIO()
    image1.save(buf1, format="PNG")
    b64_1 = base64.b64encode(buf1.getvalue()).decode("utf-8")

    buf2 = io.BytesIO()
    image2.save(buf2, format="PNG")
    b64_2 = base64.b64encode(buf2.getvalue()).decode("utf-8")

    prompt = (
        f"You will get 2 images, you should rate them based on how well they follow the positive prompt ({pos}),"
        f"and how well they avoid the negative prompt ({neg}), that means the more *unrelated* the negative prompt is to the image the better,"
        # f"and how well they avoid the negative prompt ({neg}), that means the more *unrelated* the negative prompt is to the image the better,"
        f"and also their quality. For each item you can rate from 0.0-2.0, 0 means bad and 2 means good. When the negative prompt is contradicted with positive prompt or quality, following the negative prompt should "
        f"not be a reason to decrease score for the positive and quality score. "
        f"The scoring is releative, so if image 1 is much better than image 2, image 1 should get a score higher than image 2"
    )

    completion = client.beta.chat.completions.parse(
        model="gpt-4.1",
        messages=[
            {"role": "user", "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_1}"}},
                {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_2}"}},
            ]},
        ],
        response_format=Score,
        temperature=0.0,
    )

    answer = completion.choices[0].message.parsed

    if not swapped:
      answer = np.array(((answer.first_image_positive, answer.second_image_positive), (answer.first_image_negative, answer.second_image_negative), (answer.first_image_quality, answer.second_image_quality)))
    else:
      answer = np.array(((answer.second_image_positive, answer.first_image_positive), (answer.second_image_negative, answer.first_image_negative), (answer.second_image_quality, answer.first_image_quality)))
    return answer


In [163]:
import time
import numpy as np
import cv2
import random
seed = 1989#random.randint(0, 10000)
print(seed)
win = 0
total = 0
wandb.init(project="flux")

scores = np.zeros((2, 3))

for i in prompts:
  NAGFluxPipeline._set_nag_attn_processor = _set_nag_attn_processor

  positive_prompt = i["pos"]
  # negative_prompt = "low resolution, blury, low quality, poor details"
  negative_prompt = i["neg"]# + " low resolution"
  print(negative_prompt)
  image_ours = pipe(
      positive_prompt + "high quality, 4k, trending",
      guidance_scale=0.,
      nag_negative_prompt=negative_prompt,
      nag_scale=5,
      generator=torch.Generator(device="cuda").manual_seed(seed),
      num_inference_steps=5,
      max_sequence_length=64,
  ).images[0]

  NAGFluxPipeline._set_nag_attn_processor = _set_nag_attn_processor_vanilla

  image_vanilla = pipe(
      positive_prompt + "high quality, 4k, trending",
      guidance_scale=0.,
      nag_negative_prompt=negative_prompt,
      nag_scale=4,
      generator=torch.Generator(device="cuda").manual_seed(seed),
      num_inference_steps=5,
      max_sequence_length=64,
  ).images[0]

  ans = ask_gpt(image_ours, image_vanilla, positive_prompt, negative_prompt).T
  print(ans)
  scores += ans
  full = cv2.hconcat([np.array(image_ours), np.array(image_vanilla)])
  wandb.log({"img": wandb.Image(full, caption=f"+:{positive_prompt}\n-:{negative_prompt}")})
  pylab.imshow(full)
  pylab.axis('off')
  pylab.show()



1989


modern


  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 404 Not Found"
INFO:openai._base_client:Retrying request to /chat/completions in 0.436990 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 404 Not Found"
INFO:openai._base_client:Retrying request to /chat/completions in 0.945290 seconds
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 404 Not Found"


APIConnectionError: Connection error.

In [ ]:
## compare is better score is not objective

In [ ]:
import pandas as pd

pd.DataFrame(scores, columns=["positive", "negative", "quality"], index=["ours", "vanilla"])